# Phase 2 — Monolingual IR Inference (Kaggle)

**Datasets to attach before running:**
| Kaggle Dataset name | Role |
|---|---|
| `Phase1_Dataset` | LRL-IR source code + DatasetPhase2 parquets |
| `Phase1_2000` (or 4096 / 6000 / 8000) | Phase 2 training output (model.pth + qd_test) |

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers rank-bm25 py-vncorenlp fastparquet POT python-dotenv

## 2. Paths & environment

Kaggle dataset layout (from screenshot):
```
/kaggle/input/
  phase1-dataset/
    LRL-IR/
      LRL-IR/            ← actual project root (double folder because of upload)
        DatasetPhase2/
        components/
        models/
        utils/
        ...
  phase2-model/          ← Phase 2 training output (saved from training notebook)
    doc_corpus/
    qd_test/
      qd_test.parquet
    sentence_transformer_finetune/vi/
    custom_sentence_transformer_trained/vi/
      model.pth          ← main checkpoint
```

In [ ]:
import os, sys, torch

# ── EDIT these two lines to match your attached dataset names ──────────────────
PROJECT_DATASET_NAME = "phase1-dataset"  # dataset containing LRL-IR source code
MODEL_DATASET_NAME   = "phase2-model"    # dataset containing the Phase 2 training output
# ──────────────────────────────────────────────────────────────────────────────

LANGUAGE    = "vi"
WORKING_DIR = "/kaggle/working/"

# Source code — double LRL-IR because the uploaded zip top-level folder is named LRL-IR
PROJECT_DIR = f"/kaggle/input/{PROJECT_DATASET_NAME}/LRL-IR/LRL-IR/"
DATASET_DIR = os.path.join(PROJECT_DIR, "DatasetPhase2")

# Phase 2 training output
MODEL_DIR   = f"/kaggle/input/{MODEL_DATASET_NAME}/"

# The training notebook saves the checkpoint at:
#   {OUTPUT_DIR}/custom_sentence_transformer_trained/{LANGUAGE}/model.pth
CHECKPOINT_PATH = os.path.join(MODEL_DIR, "custom_sentence_transformer_trained", LANGUAGE, "model.pth")

sys.path.insert(0, PROJECT_DIR)
os.environ["PROJECT_DIR"] = WORKING_DIR  # must end with /

print(f"PROJECT_DIR      : {PROJECT_DIR}  → exists: {os.path.isdir(PROJECT_DIR)}")
print(f"DATASET_DIR      : {DATASET_DIR}  → exists: {os.path.isdir(DATASET_DIR)}")
print(f"MODEL_DIR        : {MODEL_DIR}    → exists: {os.path.isdir(MODEL_DIR)}")
print(f"CHECKPOINT_PATH  : {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}")

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"model.pth not found at:\n  {CHECKPOINT_PATH}\n"
        f"Make sure '{MODEL_DATASET_NAME}' is attached and contains the training output."
    )

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 3. Build document corpus

`DocumentDataset._load_documents()` **always re-processes** every document on construction  
(no skip/cache logic). We write the merged corpus parquet to `/kaggle/working/` so the  
processed JSON files land in the writable directory.

In [ ]:
import pandas as pd

DOC_CORPUS_DIR = os.path.join(WORKING_DIR, "doc_corpus_infer")
PROC_DOC_DIR   = os.path.join(WORKING_DIR, "processed_docs_infer")
os.makedirs(DOC_CORPUS_DIR, exist_ok=True)
os.makedirs(PROC_DOC_DIR,   exist_ok=True)

parquet_files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")])
print(f"DatasetPhase2 parquets: {parquet_files}")

frames = []
for pf in parquet_files:
    df = pd.read_parquet(os.path.join(DATASET_DIR, pf), engine="fastparquet")
    frames.append(df[["id", "url", "title", "text"]])

full_df = (pd.concat(frames, ignore_index=True)
             .drop_duplicates(subset=["id"])
             .reset_index(drop=True))

# Write to writable dir — only the 4 columns DocumentDataset reads
full_df.to_parquet(os.path.join(DOC_CORPUS_DIR, "corpus.parquet"),
                   engine="fastparquet", index=False)
print(f"Corpus: {len(full_df)} unique documents → {DOC_CORPUS_DIR}")

# Lookup for display (str keys for safe id comparison)
id_to_row = {str(row["id"]): row for _, row in full_df.iterrows()}

## 4. Fix checkpoint path & load model

The checkpoint stores the **absolute path** of the `SentenceTransformer` directory from  
the training session. On a different Kaggle session that path doesn't exist, so we  
detect and patch it before loading the model.

In [ ]:
# Peek at the checkpoint to read the embedded SentenceTransformer path
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
st_path_in_ckpt = ckpt["sentence_transformer_save_path"]
print(f"Embedded SentenceTransformer path: {st_path_in_ckpt}")

LOAD_CHECKPOINT = CHECKPOINT_PATH  # will be overridden if path needs patching

if not os.path.isdir(st_path_in_ckpt):
    # The training notebook saves the SentenceTransformer alongside model.pth:
    #   {MODEL_DIR}/sentence_transformer_finetune/{LANGUAGE}/
    fallback = os.path.join(MODEL_DIR, "sentence_transformer_finetune", LANGUAGE)
    print(f"Path missing — trying fallback: {fallback}")
    assert os.path.isdir(fallback), (
        f"SentenceTransformer directory not found at {fallback}.\n"
        "Make sure the full training output dataset is attached."
    )
    ckpt["sentence_transformer_save_path"] = fallback
    LOAD_CHECKPOINT = os.path.join(WORKING_DIR, "model_fixed.pth")
    torch.save(ckpt, LOAD_CHECKPOINT)
    print(f"Fixed checkpoint saved → {LOAD_CHECKPOINT}")
else:
    print("SentenceTransformer path is valid — no patch needed.")

In [ ]:
from models.monolingual_retrival import MonoLingualRetrival

model = MonoLingualRetrival(
    document_dir          = DOC_CORPUS_DIR,
    processed_doc_store_dir = PROC_DOC_DIR,
    # expects the path to the model.pth FILE, not the directory
    custom_sentence_transformer_pretrained_or_save_path = LOAD_CHECKPOINT,
    language              = LANGUAGE,
    original_query_doc_count  = 30,
    extended_query_doc_count  = 30,
    chunk_length_limit    = 128,
    relevant_threshold    = 0.1,
    relevant_default_lowerbound = 0.25,
    device                = DEVICE,
    batch_size            = 32,
)
model.eval()
print("Model loaded successfully.")

## 5. Display helper

In [ ]:
def display_results(query: str, doc_ids: list, scores: list, top_n: int = 5):
    print(f"\n{'='*65}")
    print(f"Query: {query}")
    print(f"{'='*65}")
    if not doc_ids:
        print("  ⚠  No documents met the relevance threshold.")
        return
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    for rank, (doc_id, score) in enumerate(ranked[:top_n], start=1):
        row     = id_to_row.get(str(doc_id), {})
        title   = row.get("title",  "[not found]")
        snippet = str(row.get("text", ""))[:200].replace("\n", " ")
        url     = row.get("url",    "")
        print(f"\n  Rank #{rank}  |  score: {score:.4f}  |  id: {doc_id}")
        print(f"  Title  : {title}")
        print(f"  Snippet: {snippet}...")
        print(f"  URL    : {url}")
    print(f"\n  Total retrieved: {len(doc_ids)}")

## 6. Single query

In [ ]:
query = "lạm phát ảnh hưởng đến sức mua của người tiêu dùng"  # ← change this

with torch.no_grad():
    doc_ids, scores = model(query)

display_results(query, doc_ids, scores)

## 7. Batch queries

In [ ]:
queries = [
    "lạm phát ảnh hưởng đến sức mua của người tiêu dùng",
    "thị trường chứng khoán Việt Nam",
    "chính sách tiền tệ của ngân hàng nhà nước",
]

for q in queries:
    with torch.no_grad():
        doc_ids, scores = model(q)
    display_results(q, doc_ids, scores, top_n=3)

## 8. Evaluate on test split — MRR, Acc@1, Acc@5

In [ ]:
import json

# qd_test.parquet was saved by the training notebook under qd_test/
QD_TEST_PARQUET = os.path.join(MODEL_DIR, "qd_test", "qd_test.parquet")

if not os.path.exists(QD_TEST_PARQUET):
    print(f"⚠  Test parquet not found at {QD_TEST_PARQUET}")
    print("   Attach the training output dataset and check MODEL_DATASET_NAME.")
else:
    test_df = pd.read_parquet(QD_TEST_PARQUET, engine="fastparquet")
    print(f"Evaluating on {len(test_df)} test queries...")

    mrr_sum = acc1_sum = acc5_sum = 0
    details = []

    for i, (_, row) in enumerate(test_df.iterrows()):
        query   = str(row["query"])
        # id is int64 in parquet → int in JSON → compare as str across all boundaries
        true_id = str(row["id"])

        with torch.no_grad():
            doc_ids, scores = model(query)

        ranked_ids = [str(d) for d, _ in
                      sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)]

        rr = 0.0
        for rank, rid in enumerate(ranked_ids, start=1):
            if rid == true_id:
                rr = 1.0 / rank
                break

        mrr_sum  += rr
        acc1_sum += 1 if ranked_ids and ranked_ids[0] == true_id else 0
        acc5_sum += 1 if true_id in ranked_ids[:5] else 0
        details.append({"query": query, "true_id": true_id,
                        "retrieved_top5": ranked_ids[:5], "rr": rr})

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(test_df)}]  running MRR={mrr_sum/(i+1):.4f}")

    n = len(test_df)
    print("\n" + "="*40)
    print(f"MRR   : {mrr_sum/n:.4f}")
    print(f"Acc@1 : {acc1_sum/n:.4f}")
    print(f"Acc@5 : {acc5_sum/n:.4f}")

    result_path = os.path.join(WORKING_DIR, "inference_eval_results.json")
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump({"MRR": mrr_sum/n, "Acc@1": acc1_sum/n,
                   "Acc@5": acc5_sum/n, "details": details},
                  f, ensure_ascii=False, indent=2)
    print(f"Results saved → {result_path}")

## 9. Interactive loop (optional)

In [ ]:
# Uncomment to enter queries manually
# while True:
#     query = input("Enter query (blank to quit): ").strip()
#     if not query:
#         break
#     with torch.no_grad():
#         doc_ids, scores = model(query)
#     display_results(query, doc_ids, scores)